# CV 4대 태스크 실습 — ①분류 실습1: 색상 히스토그램 + k-NN

### 셀 1 — 패키지 설치 및 한글 폰트 설정

In [ ]:
!pip install -q opencv-python-headless matplotlib numpy scikit-learn scikit-image
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

import os
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import List, Optional, Sequence

import cv2
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np

# matplotlib 기본 폰트에는 한글이 없어 그래프 제목이 깨진다. 나눔고딕 폰트를 등록한다.
nanum_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if os.path.exists(nanum_path):
    fm.fontManager.addfont(nanum_path)
    plt.rcParams["font.family"] = fm.FontProperties(fname=nanum_path).get_name()
plt.rcParams["axes.unicode_minus"] = False

# 이 파일은 클린 아키텍처(Clean Architecture)에 따라 아래 4개 계층으로 구성된다.
#   1) 도메인 계층      : 알고리즘에 무관한 값 객체 / 추상 인터페이스
#   2) 유스케이스 계층   : "분류"라는 비즈니스 로직을 추상 인터페이스만으로 조립
#   3) 인프라 계층      : cv2/sklearn을 이용한 인터페이스의 실제 구현체 (어댑터)
#   4) 구성 루트        : 실제 cv2/sklearn 구현체를 생성해 유스케이스에 주입(DI)하고 실행하는 Colab 셀들
# 각 계층은 아래쪽 배너 주석으로 명확히 구분한다.


### 셀 2 — [도메인 계층 / Domain Layer] 값 객체와 추상 인터페이스

In [ ]:
# ===== [도메인 계층 / Domain Layer] =====
# 이 계층은 "특징을 추출한다", "학습해서 예측한다"는 핵심 개념만 정의하며,
# cv2/sklearn 등 구체적인 구현 기술에는 전혀 의존하지 않는다 (의존성 역전 원칙 DIP의 토대).


@dataclass
class ClassificationResult:
    """분류 결과 1건을 표현하는 값 객체(Value Object).

    예측 레이블과 (있다면) 정답 레이블을 함께 담아, 호출하는 쪽에서
    정오 판정이나 통계 계산을 쉽게 할 수 있도록 한다.
    """

    predicted_label: str
    true_label: str = ""

    @property
    def is_correct(self) -> bool:
        """정답 레이블이 주어진 경우, 예측이 맞았는지 여부."""
        return self.predicted_label == self.true_label


class FeatureExtractor(ABC):
    """이미지에서 분류용 특징 벡터를 뽑아내는 추상 인터페이스.

    SRP: "특징을 어떻게 추출할지"만 책임지며, 분류 판단이나 시각화는 담당하지 않는다.
    ISP: 다른 책임(학습/예측 등)과 섞이지 않은 작은 인터페이스로 유지한다.
    """

    @abstractmethod
    def extract(self, image_bgr: np.ndarray) -> np.ndarray:
        """BGR 이미지(np.ndarray)를 입력받아 1차원 특징 벡터(np.ndarray)를 반환한다."""
        raise NotImplementedError


class Classifier(ABC):
    """특징 벡터로 학습하고, 새로운 특징 벡터의 레이블을 예측하는 추상 인터페이스.

    SRP: "학습/예측"이라는 분류 결정 책임만 가진다.
    OCP: 새로운 분류 알고리즘은 이 인터페이스를 구현하는 새 클래스를 추가하는 것으로
         확장 가능하며, 기존 유스케이스 코드를 수정할 필요가 없다.
    """

    @abstractmethod
    def fit(self, features: np.ndarray, labels: Sequence[str]) -> None:
        """학습 데이터(특징 벡터 배열, 레이블 배열)로 분류기를 학습시킨다."""
        raise NotImplementedError

    @abstractmethod
    def predict(self, features: np.ndarray) -> List[str]:
        """특징 벡터 배열에 대한 예측 레이블 리스트를 반환한다."""
        raise NotImplementedError



### 셀 3 — [유스케이스 계층 / Use Case Layer] 분류 유스케이스

In [ ]:
# ===== [유스케이스 계층 / Use Case Layer] =====
# 이 클래스는 FeatureExtractor/Classifier라는 "추상 인터페이스"에만 의존한다 (DIP).
# cv2나 sklearn을 직접 import하거나 호출하지 않으므로, 인프라 구현체를 교체해도
# 이 클래스는 전혀 바뀌지 않는다.


class ClassificationUseCase:
    """"이미지를 분류한다"는 유스케이스(응용 비즈니스 로직)를 담당한다.

    구체적인 특징 추출 알고리즘과 분류 알고리즘은 생성자 주입(Constructor Injection)으로
    전달받는다 — 어떤 구현체를 쓸지는 이 클래스가 아니라 구성 루트(Composition Root)가 결정한다.
    """

    def __init__(self, extractor: FeatureExtractor, classifier: Classifier):
        self._extractor = extractor
        self._classifier = classifier

    def train(self, images: Sequence[np.ndarray], labels: Sequence[str]) -> None:
        """이미지 목록에서 특징을 추출한 뒤 분류기를 학습시킨다."""
        features = np.array([self._extractor.extract(img) for img in images])
        self._classifier.fit(features, labels)

    def classify_batch(
        self,
        images: Sequence[np.ndarray],
        true_labels: Optional[Sequence[str]] = None,
    ) -> List[ClassificationResult]:
        """이미지 목록을 분류해 ClassificationResult 리스트로 반환한다.

        true_labels가 주어지면 각 결과에 정답 레이블도 함께 담는다(테스트/평가용).
        """
        features = np.array([self._extractor.extract(img) for img in images])
        preds = self._classifier.predict(features)
        labels = true_labels if true_labels is not None else [""] * len(preds)
        return [
            ClassificationResult(predicted_label=p, true_label=t)
            for p, t in zip(preds, labels)
        ]

    @staticmethod
    def accuracy(results: Sequence[ClassificationResult]) -> float:
        """분류 결과 리스트로부터 정확도(0.0~1.0)를 계산한다."""
        if not results:
            return 0.0
        return sum(1 for r in results if r.is_correct) / len(results)



### 셀 4 — [인프라/어댑터 계층 / Infrastructure Layer] 히스토그램 추출기 + k-NN 어댑터

In [ ]:
# ===== [인프라/어댑터 계층 / Infrastructure Layer] =====
# cv2, sklearn 등 구체적인 외부 라이브러리를 사용하는 코드는 이 계층에만 둔다.
# 아래 두 클래스는 도메인 계층의 인터페이스를 "구현"하는 어댑터다.
from sklearn.neighbors import KNeighborsClassifier  # noqa: E402


class HSVHistogramExtractor(FeatureExtractor):
    """색상 히스토그램(HSV H·S 2D 히스토그램)을 특징으로 사용하는 FeatureExtractor 구현체.

    "분류"의 가장 단순한 고전적 접근 — 이미지 전체의 색 분포(히스토그램)를
    "지문"처럼 사용한다.

    OCP: HOG 등 다른 특징 추출 방식이 필요하면 FeatureExtractor를 구현하는
         새 클래스를 추가하면 되고, 이 클래스나 유스케이스를 고칠 필요가 없다.
    LSP: extract()의 입출력 규약이 FeatureExtractor와 완전히 동일하므로
         어디서든 FeatureExtractor 대신 안전하게 대체할 수 있다.
    """

    def __init__(self, bins: int = 32):
        self._bins = bins

    def extract(self, image_bgr: np.ndarray) -> np.ndarray:
        hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
        # H(색상), S(채도) 2개 채널에 대한 2D 히스토그램 — 밝기(V)는 조명에 민감하므로 제외
        hist = cv2.calcHist(
            [hsv], [0, 1], None, [self._bins, self._bins], [0, 180, 0, 256]
        )
        cv2.normalize(hist, hist, 0, 1, cv2.NORM_MINMAX)
        return hist.flatten()


class KNNClassifierAdapter(Classifier):
    """scikit-learn의 KNeighborsClassifier를 Classifier 인터페이스에 맞춰 감싸는 어댑터.

    LSP: fit()/predict()의 시그니처와 동작이 Classifier 인터페이스 계약을 그대로
         만족하므로, Classifier를 기대하는 어떤 코드에도 안전하게 대입할 수 있다.
    """

    def __init__(self, n_neighbors: int = 3, metric: str = "euclidean"):
        self._model = KNeighborsClassifier(n_neighbors=n_neighbors, metric=metric)

    def fit(self, features: np.ndarray, labels: Sequence[str]) -> None:
        self._model.fit(features, labels)

    def predict(self, features: np.ndarray) -> List[str]:
        return list(self._model.predict(features))



### 셀 5 — [구성 루트 / Composition Root] 실습용 샘플 이미지 준비 (3개 클래스: 고양이 / 사람 / 커피잔)

In [ ]:
# ===== [구성 루트 / Composition Root — Colab 실행 코드] =====
# 이 아래부터는 실제 cv2/sklearn 구현체를 생성하고, 유스케이스에 주입(DI)해서
# 실행 · 시각화 · 파일 저장까지 수행하는 "조립" 코드다. 이 계층만 파일 I/O와
# matplotlib 시각화, !pip 설치를 수행한다.
os.makedirs("/content/samples", exist_ok=True)

# 위키미디어 공개 이미지 대신, scikit-image에 내장된 표준 샘플 이미지를 사용한다
# (별도 다운로드 없이 어떤 환경에서도 동일하게 재현 가능)
from skimage import data as skdata  # noqa: E402

CLASSES = ["cat", "person", "coffee"]
base_images = {
    "cat": skdata.chelsea(),  # 고양이 사진
    "person": skdata.astronaut(),  # 사람(우주비행사) 사진
    "coffee": skdata.coffee(),  # 커피잔 사진
}
for name, arr in base_images.items():
    bgr = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
    cv2.imwrite(f"/content/samples/{name}_base.jpg", bgr)

print("기준 이미지 3장 준비 완료:", list(base_images.keys()))



### 셀 6 — [구성 루트] 데이터 증강(augmentation) — 클래스마다 여러 장의 학습 샘플을 합성

In [ ]:
# 기준 이미지 1장만으로는 "분류기"를 학습하기엔 샘플이 부족하므로,
# 밝기/대비를 살짝 바꾸고 좌우반전/크롭한 변형본들을 만들어 학습셋을 늘린다.
# (분류 알고리즘 자체와는 무관한, 실습용 테스트 데이터 생성 로직이므로 구성 루트에 둔다.)
def augment(img_bgr, n=6, seed=0):
    rng = np.random.default_rng(seed)
    h, w = img_bgr.shape[:2]
    outs = []
    for i in range(n):
        out = img_bgr.copy()
        if rng.random() < 0.5:
            out = cv2.flip(out, 1)
        alpha = 0.8 + 0.4 * rng.random()  # 대비
        beta = rng.integers(-20, 20)  # 밝기
        out = cv2.convertScaleAbs(out, alpha=alpha, beta=beta)
        # 랜덤 크롭 후 원래 크기로 리사이즈 (약간의 위치/스케일 변화)
        ch, cw = int(h * 0.85), int(w * 0.85)
        y0 = rng.integers(0, h - ch + 1)
        x0 = rng.integers(0, w - cw + 1)
        out = out[y0 : y0 + ch, x0 : x0 + cw]
        out = cv2.resize(out, (w, h))
        outs.append(out)
    return outs



### 셀 7 — [구성 루트] 의존성 주입(DI) — 구체 어댑터를 생성해 유스케이스에 조립하고 학습

In [ ]:
# 여기서만 구체 클래스(HSVHistogramExtractor, KNNClassifierAdapter)를 알고 있다.
# 유스케이스/도메인 계층은 이 구체 타입들의 존재를 전혀 모른다.
extractor = HSVHistogramExtractor(bins=32)
classifier = KNNClassifierAdapter(n_neighbors=3, metric="euclidean")
use_case = ClassificationUseCase(extractor=extractor, classifier=classifier)

train_images, train_y = [], []
for label in CLASSES:
    base = cv2.imread(f"/content/samples/{label}_base.jpg")
    variants = augment(base, n=8, seed=hash(label) % 1000)
    for v in variants:
        train_images.append(v)
        train_y.append(label)

use_case.train(train_images, train_y)
print("학습 샘플 개수:", len(train_images), "| 클래스당:", len(train_images) // len(CLASSES))
print("k-NN 분류기 학습 완료 (k=3)")


### 셀 8 — [구성 루트] 테스트 — 학습에 쓰지 않은 새로운 변형 이미지로 분류 정확도 확인

In [ ]:
test_images, test_y = [], []
for label in CLASSES:
    base = cv2.imread(f"/content/samples/{label}_base.jpg")
    variants = augment(base, n=4, seed=hash(label) % 1000 + 777)  # 학습과 다른 seed
    for v in variants:
        test_images.append(v)
        test_y.append(label)

results = use_case.classify_batch(test_images, test_y)
acc = ClassificationUseCase.accuracy(results)
pred_y = [r.predicted_label for r in results]
print(f"테스트 정확도: {acc * 100:.1f}%  ({len(test_y)}개 샘플)")


### 셀 9 — [구성 루트] 결과 시각화 — 테스트 이미지와 예측 레이블을 함께 출력

In [ ]:
fig, axes = plt.subplots(len(CLASSES), 4, figsize=(12, 9))
for i, img in enumerate(test_images):
    r, c = i // 4, i % 4
    ax = axes[r][c]
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    correct = results[i].is_correct
    color = "green" if correct else "red"
    ax.set_title(f"정답:{test_y[i]} / 예측:{pred_y[i]}", color=color, fontsize=10)
    ax.axis("off")
plt.suptitle(f"색상 히스토그램 + k-NN 분류 결과 (정확도 {acc*100:.0f}%)", fontsize=14)
plt.tight_layout()
plt.savefig("/content/classification_knn_result.png", dpi=120, bbox_inches="tight")
plt.show()
print("저장 완료: /content/classification_knn_result.png")
